# Libraries

In [1]:
# Reinstall PyTorch stack with CUDA 12.8 support
!pip uninstall -y torch torchvision torchaudio torchao
!pip install -q \
    torch==2.10.0+cu128 \
    torchvision==0.25.0+cu128 \
    torchaudio==2.10.0+cu128 \
    torchao==0.10.0 \
    --index-url https://download.pytorch.org/whl/cu128

# vLLM
!pip install -q vllm==0.18.0

# Dependencies
!pip install -q "protobuf>=5.26.1,<6.0.0" --break-system-packages

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 916.9/916.9 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 98.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 59.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87

In [2]:
# Standard library imports
import os
import gc
import json
import time
import random
import shutil
import subprocess

# Third-party libraries
import numpy as np
from vllm import LLM, SamplingParams
from huggingface_hub import login
from transformers import AutoTokenizer, set_seed
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# Deep learning framework
import torch

2026-05-21 23:21:46.417112: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779405706.646874      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779405706.709052      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779405707.208098      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779405707.208141      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779405707.208143      57 computation_placer.cc:177] computation placer alr

# Global Variables

In [3]:
# User & Repository Configuration
USER_NAME = "Abdelrahmanemam01"
EMAIL     = "abdulrahmanemam01@gmail.com"
REPO      = "MahmoudOsama20/EdgeCompress"  # Target repository

# Model Selection
MODEL = "Phi-4-mini-instruct"

# Model & Tokenizer Configuration
MODEL_ID     = "microsoft/Phi-4-mini-instruct"
TOKENIZER_ID = "microsoft/Phi-4-mini-instruct"

MODEL_NAME           = "Phi-4-mini-instruct"
MODEL_NAME_IN_REPO   = "Phi-4-mini-instruct-Original"
#COMPRESSION_METHOD   = "WANDA"
#MODEL_TYPE           = "Pruning"
#SPARSITY = 70
#PATH = f"Models/{SPARSITY}"

# Inference Prompt (Chat Format)
PROMPT = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Provide a concise explanation of Artificial Intelligence."
    }
]

dummy_prompt = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Give me story of a brave man"
    }
]

# Generation Configuration
MAX_GENERATION_TOKENS = 150
SEED = 42

# Output Configuration
OUTPUT_FILE = "model_metrics.json"

# Functions

In [4]:
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Seeding

In [5]:
set_reproducibility(SEED)

# Huggingface Login

In [6]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# GitHub login

In [7]:
token = UserSecretsClient().get_secret("GIT_TOKEN")
print("Logging into GitHub...")

Logging into GitHub...


# Download Model

In [8]:
"""local_path = snapshot_download(
    repo_id=MODEL_ID,
    allow_patterns=f"{PATH}/*",
    local_dir="/kaggle/working/model"
)"""

'local_path = snapshot_download(\n    repo_id=MODEL_ID,\n    allow_patterns=f"{PATH}/*",\n    local_dir="/kaggle/working/model"\n)'

# Prompt Preprocessing

**DummyPrompt Tokenization**

In [9]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

#Chat Formating
dummy_prompt_text = tokenizer.apply_chat_template(
    dummy_prompt, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
dummy_prompt_token_ids = tokenizer.encode(dummy_prompt_text)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

**RealPrompt Tokenization**

In [10]:
#Chat Formating
prompt_text = tokenizer.apply_chat_template(
    PROMPT, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
prompt_token_ids = tokenizer.encode(prompt_text)

# Initializing vLLM

In [11]:
llm = LLM(
    model=MODEL_ID,
    tokenizer=TOKENIZER_ID,
    dtype="float16",
    max_model_len=256,
    tensor_parallel_size=2,
    gpu_memory_utilization=0.85,
    attention_backend = "TRITON_ATTN",
    disable_log_stats=False,
    enable_prefix_caching = False
)
print("vLLM Initialized Successfully!")


sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_GENERATION_TOKENS,
    min_tokens=MAX_GENERATION_TOKENS,
    seed = SEED
)

ttft_sampling_params = SamplingParams(
    temperature=0.0,
    seed = SEED,
    max_tokens=1,
    ignore_eos=True # Ensures it doesn't accidentally stop early
)

INFO 05-21 23:22:14 [utils.py:233] non-default args: {'tokenizer': 'microsoft/Phi-4-mini-instruct', 'dtype': 'float16', 'max_model_len': 256, 'tensor_parallel_size': 2, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'attention_backend': 'TRITON_ATTN', 'model': 'microsoft/Phi-4-mini-instruct'}


config.json: 0.00B [00:00, ?B/s]

INFO 05-21 23:22:15 [config.py:437] Replacing legacy 'type' key with 'rope_type'
INFO 05-21 23:22:36 [model.py:533] Resolved architecture: Phi3ForCausalLM
WARNING 05-21 23:22:36 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 05-21 23:22:36 [model.py:1582] Using max model len 256
INFO 05-21 23:22:36 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-21 23:22:36 [vllm.py:754] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

WARNING 05-21 23:22:37 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


...

(Worker_TP0 pid=257) INFO 05-21 23:23:59 [default_loader.py:384] Loading weights took 10.70 seconds
(Worker_TP0 pid=257) INFO 05-21 23:24:00 [gpu_model_runner.py:4566] Model loading took 3.63 GiB memory and 43.746010 seconds
(Worker_TP0 pid=257) INFO 05-21 23:24:11 [backends.py:988] Using cache directory: /root/.cache/vllm/torch_compile_cache/1e79882f3c/rank_0_0/backbone for vLLM's torch.compile
(Worker_TP0 pid=257) INFO 05-21 23:24:11 [backends.py:1048] Dynamo bytecode transform time: 10.08 s


(Worker_TP0 pid=257) [rank0]:W0521 23:24:12.478000 257 torch/_inductor/utils.py:1679] Not enough SMs to use max_autotune_gemm mode
(Worker_TP1 pid=258) [rank1]:W0521 23:24:12.488000 258 torch/_inductor/utils.py:1679] Not enough SMs to use max_autotune_gemm mode


(Worker_TP0 pid=257) INFO 05-21 23:24:17 [backends.py:371] Cache the graph of compile range (1, 8192) for later use
(Worker_TP1 pid=258) INFO 05-21 23:24:17 [backends.py:371] Cache the graph of compile range (1, 8192) for later use
(Worker_TP0 pid=257) INFO 05-21 23:24:23 [backends.py:387] Compiling a graph for compile range (1, 8192) takes 11.97 s
(Worker_TP0 pid=257) INFO 05-21 23:24:24 [decorators.py:627] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/f87fbe634706e58f23ddaf7a9d073e0ea21c155cfb734ae1e6c2701b243bc455/rank_0_0/model
(Worker_TP0 pid=257) INFO 05-21 23:24:24 [monitor.py:48] torch.compile took 24.02 s in total
(Worker_TP0 pid=257) INFO 05-21 23:24:27 [monitor.py:76] Initial profiling/warmup run took 2.50 s
(Worker_TP1 pid=258) INFO 05-21 23:24:37 [kv_cache_utils.py:826] Overriding num_gpu_blocks=0 with num_gpu_blocks_override=512
(Worker_TP1 pid=258) INFO 05-21 23:24:37 [gpu_model_runner.py:5607] Profiling CUDA graph memory: PIECEWI

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 11.82it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:06<00:00,  5.36it/s]


(Worker_TP0 pid=257) INFO 05-21 23:24:55 [custom_all_reduce.py:216] Registering 5590 cuda graph addresses
(Worker_TP1 pid=258) INFO 05-21 23:24:55 [custom_all_reduce.py:216] Registering 5590 cuda graph addresses
(Worker_TP0 pid=257) INFO 05-21 23:24:56 [gpu_model_runner.py:5746] Graph capturing finished in 12 secs, took 0.46 GiB
(Worker_TP0 pid=257) INFO 05-21 23:24:56 [gpu_worker.py:617] CUDA graph pool memory: 0.46 GiB (actual), 0.49 GiB (estimated), difference: 0.03 GiB (5.9%).
(Worker_TP1 pid=258) INFO 05-21 23:24:56 [gpu_worker.py:617] CUDA graph pool memory: 0.46 GiB (actual), 0.49 GiB (estimated), difference: 0.03 GiB (5.9%).
(EngineCore pid=232) INFO 05-21 23:24:56 [core.py:281] init engine (profile, create kv cache, warmup model) took 55.65 seconds
(EngineCore pid=232) INFO 05-21 23:24:59 [vllm.py:754] Asynchronous scheduling is enabled.
INFO 05-21 23:24:59 [llm.py:391] Supported tasks: ['generate']
vLLM Initialized Successfully!


# Inference

In [12]:
# ── Initialize collectors ──────────────────────────────────────────
ttft_times    = []
latency_times = []

# ── Warm-up ONCE, outside the measurement loop ────────────────────
for _ in range(5):
    llm.generate(
        prompts=[{"prompt_token_ids": dummy_prompt_token_ids}],
        sampling_params=SamplingParams(max_tokens=50, temperature=0.0, seed=SEED),
        use_tqdm=False,
    )

# ── Measurement loop ──────────────────────────────────────────────
N_RUNS = 30
for _ in range(N_RUNS):
    output  = llm.generate(
        prompts=[{"prompt_token_ids": prompt_token_ids}],
        sampling_params=sampling_params,
        use_tqdm=False,
    )
    metrics = output[0].metrics

    # Extract from RequestStateStats fields
    ttft    = metrics.first_token_latency                    # prefill cost
    latency = metrics.last_token_ts - metrics.queued_ts      # total e2e latency

    ttft_times.append(ttft)
    latency_times.append(latency)

# ── Stats ─────────────────────────────────────────────────────────
ttft_arr,     latency_arr    = np.array(ttft_times), np.array(latency_times)

ttft_mean,    ttft_std       = ttft_arr.mean(),    ttft_arr.std()
latency_mean, latency_std    = latency_arr.mean(), latency_arr.std()

# decode throughput: excludes first token, over decode-only window
throughput_arr               = (MAX_GENERATION_TOKENS - 1) / (latency_arr - ttft_arr)
throughput_mean, throughput_std = throughput_arr.mean(), throughput_arr.std()

input_tokens           = len(prompt_token_ids)
total_generated_tokens = MAX_GENERATION_TOKENS

INFO 05-21 23:25:11 [loggers.py:259] Engine 000: Avg prompt throughput: 10.7 tokens/s, Avg generation throughput: 45.9 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%
INFO 05-21 23:25:21 [loggers.py:259] Engine 000: Avg prompt throughput: 7.6 tokens/s, Avg generation throughput: 52.5 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%
INFO 05-21 23:25:31 [loggers.py:259] Engine 000: Avg prompt throughput: 5.7 tokens/s, Avg generation throughput: 46.3 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%
INFO 05-21 23:25:41 [loggers.py:259] Engine 000: Avg prompt throughput: 5.7 tokens/s, Avg generation throughput: 40.2 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
INFO 05-21 23:25:51 [loggers.py:259] Engine 000: Avg prompt throughput: 3.8 tokens/s, Avg generation throughput: 35.0 tokens/s, Running

In [13]:
print(latency_times)

[2.736154856999974, 2.75983402199995, 2.8011636690000614, 2.8274737899998854, 2.8651966259999426, 3.0102606240000114, 3.113586798000142, 3.306572813999992, 3.55009958200003, 3.721038086000135, 3.7414136759998655, 3.9634871060000023, 4.302147950999824, 4.58044185100016, 6.493066658999851, 9.305836565000163, 9.165822061999961, 9.565561002999857, 7.548010651000141, 5.101356976000034, 4.928947446999928, 4.348112295999954, 4.248364008999943, 4.237721246999854, 4.278821774000107, 4.417017117999876, 4.697671882000122, 4.699803426000017, 5.663945804999912, 5.651472832999843]


In [14]:
print(ttft_times)

[0.03649139404296875, 0.033098697662353516, 0.03277158737182617, 0.0323796272277832, 0.03410077095031738, 0.03485274314880371, 0.03296995162963867, 0.033835411071777344, 0.03461480140686035, 0.035691022872924805, 0.03291940689086914, 0.03773665428161621, 0.03549337387084961, 0.0399472713470459, 0.03567314147949219, 0.0435795783996582, 0.04687643051147461, 0.04699277877807617, 0.05383872985839844, 0.039359092712402344, 0.03818655014038086, 0.03588724136352539, 0.03410983085632324, 0.04005932807922363, 0.03426241874694824, 0.03375840187072754, 0.037084102630615234, 0.037294626235961914, 0.036692142486572266, 0.03376936912536621]


# Results

**Writing in json file**

In [15]:
benchmark_results = {
    "model_name"            : MODEL_NAME,
    #"model_type"            : MODEL_TYPE,
    #"compression_method"    : COMPRESSION_METHOD,
    #"sparsity"              : SPARSITY,
    "input_token_count"     : input_tokens,
    "generated_token_count" : total_generated_tokens,
    "ttft_sec"              : f"{ttft_mean:.2f} ± {ttft_std:.2f}",
    "latency_sec"           : f"{latency_mean:.2f} ± {latency_std:.2f}",
    "throughput_tokens_per_sec" : f"{throughput_mean:.2f} ± {throughput_std:.2f}",
}


# Save Results
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(benchmark_results, file, indent=4, ensure_ascii=False)

print(f"[INFO] Metrics successfully saved to '{OUTPUT_FILE}'")

[INFO] Metrics successfully saved to 'model_metrics.json'


**Push Results to GitHub Repository**

In [16]:
# Paths & Repository Setup
target_dir_in_repo = f"Evaluations/InferenceEvaluations/CloudEvaluation/{MODEL}/{MODEL_NAME_IN_REPO}"
source_file = OUTPUT_FILE 
remote_url = f"https://{token}@github.com/{REPO}.git"


# Local temporary directory
current_dir = os.getcwd()
temp_repo_dir = os.path.join(current_dir, "temp_git_repo")


# Clean Previous Runs
if os.path.exists(temp_repo_dir):
    shutil.rmtree(temp_repo_dir)


# Clone Repository
print(f"Cloning repository into: {temp_repo_dir}")
subprocess.run(
    ["git", "clone", remote_url, temp_repo_dir],
    check=True
)


# Prepare Target Directory
full_target_path = os.path.join(temp_repo_dir, target_dir_in_repo)
os.makedirs(full_target_path, exist_ok=True)


# Copy Source File (FIXED)
if os.path.exists(source_file):
    dest_path = os.path.join(full_target_path, os.path.basename(source_file))
    shutil.copy2(source_file, dest_path)
else:
    print(f"Warning: Source file '{source_file}' does not exist.")


# Git Config, Commit & Push
try:
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.email", EMAIL],
        check=True
    )
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.name", USER_NAME],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "add", "."],
        check=True
    )

    commit_msg = f"Added the cloud inference evaluation results to {target_dir_in_repo}"
    subprocess.run(
        ["git", "-C", temp_repo_dir, "commit", "-m", commit_msg],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "push", "origin", "main"],
        check=True
    )

    print(f"Successfully pushed to '{REPO}' → '{target_dir_in_repo}'")

except subprocess.CalledProcessError as error:
    print(f"Git operation failed: {error}")

Cloning repository into: /kaggle/working/temp_git_repo


Cloning into '/kaggle/working/temp_git_repo'...


[main 01e6030] Added the cloud inference evaluation results to Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-Original
 1 file changed, 8 insertions(+)
 create mode 100644 Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-Original/model_metrics.json
Successfully pushed to 'MahmoudOsama20/EdgeCompress' → 'Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-Original'


To https://github.com/MahmoudOsama20/EdgeCompress.git
   7667863..01e6030  main -> main
